# Mock 03 — fixed version

Same question, done point-in-time correctly. Each fix is marked **Fix N** and numbered as in
`mock_03_solution.md`. Frame is indexed by the *target* hour `T`; the decision time is 12:00 UTC
on the day before `T`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

df = pd.read_csv("../../data/hourly_power_clean.csv", parse_dates=["time"])
# Fix 3: parse BOTH timestamps; origin_datetime is the column that decides availability
fc = pd.read_csv("../../data/weather_forecasts.csv", parse_dates=["origin_datetime", "forecast_datetime"])
print(fc.dtypes)
print("forecasts per target hour:", fc.groupby("forecast_datetime").size().value_counts().to_dict())
print("horizons:", fc["horizon_h"].min(), "to", fc["horizon_h"].max(), "hours")

origin_datetime      datetime64[ns, UTC]
forecast_datetime    datetime64[ns, UTC]
horizon_h                          int64
temp_forecast_c                  float64
dtype: object
forecasts per target hour: {4: 17483, 1: 12, 2: 12, 3: 12}
horizons: 1 to 48 hours


**Fix 7 / 11 — build the frame around the target hour and the decision time.** For a target hour
`T` on day D+1 the decision is taken at 12:00 UTC on day D. Consumption at `T-24h` (same hour on
day D) is *not* known yet for hours after 12:00, so the persistence feature must be `T-48h`
(and `T-168h`). Actual temperature at `T` is kept only as a hindsight benchmark.

In [2]:
t = df.rename(columns={"time": "T", "consumption_mwh": "y", "temp_c": "temp_actual"})[["T", "y", "temp_actual"]].copy()
t["decision_time"] = t["T"].dt.floor("D") - pd.Timedelta(hours=12)

# Fix 4: derive local hour from the tz-aware timestamp; never strip tz before joining
t["hour"] = t["T"].dt.tz_convert("Europe/London").dt.hour
t["weekend"] = (t["T"].dt.dayofweek >= 5).astype(int)

t["cons_lag48"] = t["y"].shift(48)
t["cons_lag168"] = t["y"].shift(168)

# what the broken notebook used, kept only to show its effect
t["cons_lag24_unavailable"] = t["y"].shift(24)

t[["T", "decision_time", "hour", "cons_lag48", "cons_lag168"]].iloc[200:203]

,T,decision_time,hour,cons_lag48,cons_lag168
200,2022-01-09 08:00:00+00:00,2022-01-08 12:00:00+00:00,8,35603.0,32164.7
201,2022-01-09 09:00:00+00:00,2022-01-08 12:00:00+00:00,9,37243.0,32584.5
202,2022-01-09 10:00:00+00:00,2022-01-08 12:00:00+00:00,10,37327.7,32173.9


**Fix 2 — choose the forecast that existed at decision time.** Keep only forecasts whose
`origin_datetime <= decision_time` of their target hour, then take the most recent of those.
The resulting horizons are 12–35 h, never 1–12 h.

In [3]:
fc["decision_time"] = fc["forecast_datetime"].dt.floor("D") - pd.Timedelta(hours=12)
available = fc[fc["origin_datetime"] <= fc["decision_time"]]

honest = (
    available.sort_values("origin_datetime")
             .groupby("forecast_datetime")
             .last()[["temp_forecast_c", "horizon_h", "origin_datetime"]]
)

# the broken notebook's choice, for comparison
latest_any = (
    fc.sort_values("origin_datetime")
      .groupby("forecast_datetime")
      .last()[["temp_forecast_c", "horizon_h"]]
      .rename(columns={"temp_forecast_c": "temp_fc_latest_any", "horizon_h": "horizon_latest_any"})
)

print("honest horizons:", honest["horizon_h"].min(), "-", honest["horizon_h"].max())
print("broken 'latest' horizons:", latest_any["horizon_latest_any"].min(), "-", latest_any["horizon_latest_any"].max())

honest horizons: 12 - 35
broken 'latest' horizons: 1 - 12


In [4]:
# Fix 8: one-to-one merge on tz-aware UTC keys, validated; row count must not change
n_before = len(t)
t = t.merge(honest, left_on="T", right_index=True, how="left", validate="one_to_one")
t = t.merge(latest_any, left_on="T", right_index=True, how="left", validate="one_to_one")
assert len(t) == n_before
assert (t["origin_datetime"].dropna() <= t.loc[t["origin_datetime"].notna(), "decision_time"]).all()

# Fix 10: no bfill — a forecast missing at decision time stays missing (first 2 days) and the row is dropped
t = t.dropna()
print(len(t), "rows")

17352 rows


**Fix 5 / 6 — look at the forecast before using it.** Bias and error growth with horizon.

In [5]:
err = t["temp_forecast_c"] - t["temp_actual"]
print(f"mean forecast error (bias): {err.mean():+.2f} C   sd: {err.std():.2f} C")

by_h = err.groupby(t["horizon_h"]).agg(bias="mean", sd="std", n="size").round(2)
by_h.iloc[::4]

mean forecast error (bias): +0.31 C   sd: 1.86 C


,bias,sd,n
horizon_h,,,
12.0,0.27,1.11,723
16.0,0.31,1.34,723
20.0,0.30,1.56,723
24.0,0.26,1.84,723
28.0,0.48,2.14,723
32.0,0.32,2.33,723


In [6]:
err_any = t["temp_fc_latest_any"] - t["temp_actual"]
print(f"error sd, honest 12-35h forecast: {err.std():.2f} C")
print(f"error sd, broken 'latest' 1-12h : {err_any.std():.2f} C   <- this is why the broken notebook saw no loss")

error sd, honest 12-35h forecast: 1.86 C
error sd, broken 'latest' 1-12h : 0.82 C   <- this is why the broken notebook saw no loss


**Fix 7 — chronological split. Fix 9 — report test metrics only.**

In [7]:
split = int(len(t) * 0.8)
train, test = t.iloc[:split], t.iloc[split:]
print("train:", train["T"].min().date(), "->", train["T"].max().date())
print("test: ", test["T"].min().date(), "->", test["T"].max().date())

hour_train = pd.get_dummies(train["hour"], prefix="h").astype(float)
hour_test = pd.get_dummies(test["hour"], prefix="h").astype(float).reindex(columns=hour_train.columns, fill_value=0.0)


def fit_eval(extra_cols, lag_cols=("cons_lag48", "cons_lag168")):
    cols = list(lag_cols) + ["weekend"] + list(extra_cols)
    Xtr = pd.concat([hour_train, train[cols]], axis=1)
    Xte = pd.concat([hour_test, test[cols]], axis=1)
    m = LinearRegression().fit(Xtr, train["y"])
    p = m.predict(Xte)
    return {"R2_test": r2_score(test["y"], p),
            "MAE_test": mean_absolute_error(test["y"], p),
            "R2_train": m.score(Xtr, train["y"])}


rows = {
    "actual temp (hindsight, not available)": fit_eval(["temp_actual"]),
    "honest forecast (origin <= 12:00 D)":     fit_eval(["temp_forecast_c"]),
    "'latest' forecast (leaks 1-12h)":         fit_eval(["temp_fc_latest_any"]),
    "no temperature":                          fit_eval([]),
    "broken lag24 + actual temp":              fit_eval(["temp_actual"], lag_cols=("cons_lag24_unavailable", "cons_lag168")),
}
res = pd.DataFrame(rows).T.round(4)
res

train: 2022-01-08 -> 2023-08-09
test:  2023-08-09 -> 2023-12-31


,R2_test,MAE_test,R2_train
"actual temp (hindsight, not available)",0.9346,799.9580,0.9380
honest forecast (origin <= 12:00 D),0.9269,841.9177,0.9317
'latest' forecast (leaks 1-12h),0.9325,813.5112,0.9365
no temperature,0.9021,973.7621,0.8992
broken lag24 + actual temp,0.9355,790.4348,0.9400


## Honest result

The deployable model is "honest forecast". The gap to hindsight temperature is small in R² terms
but real in MAE, and the noon forecast is the *only* forecast we actually have at 12:00 — the
"latest forecast" row is not an option, it is a leak. The broken notebook's headline
("0.1% loss, use the latest forecast") compared three in-sample numbers computed on a randomly
split, row-duplicated frame in which the persistence feature was itself unavailable.

In [8]:
gap = res.loc["actual temp (hindsight, not available)", "R2_test"] - res.loc["honest forecast (origin <= 12:00 D)", "R2_test"]
print(f"R2 gap hindsight vs honest forecast (test): {gap:.4f}")
print(f"MAE honest forecast: {res.loc['honest forecast (origin <= 12:00 D)', 'MAE_test']:.0f} MWh  vs  no temperature: {res.loc['no temperature', 'MAE_test']:.0f} MWh")

R2 gap hindsight vs honest forecast (test): 0.0077
MAE honest forecast: 842 MWh  vs  no temperature: 974 MWh
